# Training trigger input-domain review

This review reads the full statements of the 19 remaining training tasks and the 380 saved trigger inputs. It checks input admissibility only, without executing candidate code, calling a model, inspecting held-out tasks, or selecting by outcomes. Numeric fields use whitespace-separated integer tokens; repeated spaces/tabs are admissible. Grid rows remain contiguous strings. Duplicate inputs are reported but preserved.

A result is `valid` only when every input constraint listed in the stored statement is implemented here; `invalid` names a concrete failed constraint; `unresolved` is reserved for unsupported tasks or missing semantic coverage. Unsatisfiable game instances, impossible routes, duplicate stones, and repeated candy routes are valid where the statement permits them. Graph stability/connectivity and distinct university/government nodes are checked explicitly. No remaining statement imposes an additional input-semantic promise beyond these validators. The malformed recurrence typesetting in task3790 affects task interpretation, not its explicit input bounds.

The rules are assistant-authored and reviewed against stored statements, with synthetic positive and negative cases and extra structural edge cases. This is not independent human verification of ground-truth attack success or solver correctness. Original trigger artifacts are retained; the output maps each candidate to valid/invalid/unresolved original input indexes.


In [ ]:
from pathlib import Path
import hashlib, json, os, re, sys
from collections import Counter
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
assert (REPO / "pipeline").is_dir()
from pipeline.data import Dataset, load_records
DATA = Path("data/azure_pbt_train19_s300_v1.json")
RUN = "azure-terra-pbt-train19-s300-v1-triggers"
dataset = Dataset.load(DATA)
rows = load_records(RUN)
assert len(dataset.train) == 19 and not dataset.test
assert len(rows) == 38 and all(not row["failed"] for row in rows)
assert all(row["split"] == "train" and len(row["inputs"]) == 10 for row in rows)
assert {row["candidate_id"] for row in rows} == {c.candidate_id for _, c in dataset.candidates()}
assert len({row["candidate_id"] for row in rows}) == len(rows)


In [ ]:
class DomainInvalid(ValueError):
    pass

def require(condition, reason):
    if not condition:
        raise DomainInvalid(reason)

def numbers(text):
    tokens = text.split()
    require(all(re.fullmatch(r"[+-]?[0-9]+", token) for token in tokens), "non-integer token")
    return [int(token) for token in tokens]

def bounded(values, low, high, field):
    require(all(low <= value <= high for value in values), f"{field} outside {low}..{high}")

def array_payload(values, header, n, low, high):
    require(len(values) == header + n, f"expected {n} array values, found {len(values)-header}")
    bounded(values[header:], low, high, "array element")

def graph_components(n, edge_values):
    require(len(edge_values) % 2 == 0, "incomplete edge")
    parent = list(range(n + 1))
    def find(vertex):
        while parent[vertex] != vertex:
            parent[vertex] = parent[parent[vertex]]
            vertex = parent[vertex]
        return vertex
    edges = set()
    for a, b in zip(edge_values[::2], edge_values[1::2]):
        bounded((a, b), 1, n, "edge vertex")
        require(a != b, "self-loop")
        edge = tuple(sorted((a, b)))
        require(edge not in edges, "duplicate undirected edge")
        edges.add(edge)
        parent[find(a)] = find(b)
    return [find(vertex) for vertex in range(1, n + 1)]

RULES = {
    "3694": "1<=n<=100000; n integers in0..1e9; duplicates explicitly allowed",
    "3733": "1<=n<=400000; 1<=I<=1e8; n integers in0..1e9",
    "3748": "1<=H,W<=12; exactlyH lowercase rows of widthW",
    "3771": "2<=H,W<=100; grid alphabet .oST; exactlyone S andone T",
    "3782": "1<=N<=2000;1<=K<=N;1<=Q<=N-K+1;N integers in1..1e9",
    "3790": "1<=n<=100;1<=T<=1e7;n integers in1..300",
    "3798": "exactlytwo integers n,s in1..1e11; s>n is allowed",
    "3801": "1<=n,m<=50;n binary likes with at leastone1;n weights in1..50",
    "3825": "exactlyone integer n in1..1e9",
    "3868": "1<=n<=1e5;0<=m<=1e5;1<=k<=1e6;m flights with day/cost1..1e6,cities0..n,exactlyone zero city",
    "3892": "2<=n<=100;1<=m<=200;m routepairs in1..n with distinct endpoints; duplicate routes allowed",
    "3897": "1<=n<=500;n integers in1..1e9",
    "3901": "1<=n<=2000;n integers in1..1e9",
    "3902": "one lowercase string of length5..10000",
    "3915": "two integers p,q in1..10000;q<=42p and p<=42q",
    "3929": "two integers N,K with1<=K<=N<=2000",
    "3957": "2<=n<=200000;1<=k<=floor(n/2);2k distinct university nodes; n-1 simple connected edges",
    "3977": "1<=n<=1000;0<=m<=100000;1<=k<=n;distinct governments;simple graph;no government pair connected",
    "3999": "6<=N<=400;exactly4N colors in0..999; identical tiles allowed",
}
ARRAY_RULES = {"3694": (100000, 0), "3897": (500, 1), "3901": (2000, 1)}

def check_domain(task_id, text):
    if task_id not in RULES:
        return {"status": "unresolved", "reason": "no complete validator for this task"}
    if not isinstance(text, str):
        return {"status": "invalid", "reason": "stdio input must be a string"}
    try:
        tokens = text.split()
        require(bool(tokens), "empty input")
        if task_id == "3902":
            require(len(tokens) == 1, "expected one string")
            require(5 <= len(tokens[0]) <= 10000, "string length outside5..10000")
            require(bool(re.fullmatch("[a-z]+", tokens[0])), "non-lowercase-English character")
        elif task_id in ("3748", "3771"):
            require(len(tokens) >= 2, "missing grid dimensions")
            h, w = numbers(" ".join(tokens[:2]))
            low, high = (1, 12) if task_id == "3748" else (2, 100)
            bounded((h, w), low, high, "grid dimension")
            grid = tokens[2:]
            require(len(grid) == h, "wrong grid row count")
            require(all(len(row) == w for row in grid), "wrong grid width")
            alphabet = set("abcdefghijklmnopqrstuvwxyz") if task_id == "3748" else set(".oST")
            require(all(set(row) <= alphabet for row in grid), "invalid grid character")
            if task_id == "3771":
                require("".join(grid).count("S") == 1, "expected exactly one S")
                require("".join(grid).count("T") == 1, "expected exactly one T")
        else:
            v = numbers(text)
            if task_id in ARRAY_RULES:
                nmax, amin = ARRAY_RULES[task_id]
                n = v[0]
                bounded((n,), 1, nmax, "n")
                array_payload(v, 1, n, amin, 10**9)
            elif task_id == "3733":
                n, disk = v[:2]
                bounded((n,), 1, 400000, "n")
                bounded((disk,), 1, 10**8, "I")
                array_payload(v, 2, n, 0, 10**9)
            elif task_id == "3782":
                n, k, q = v[:3]
                bounded((n,), 1, 2000, "N")
                bounded((k,), 1, n, "K")
                bounded((q,), 1, n-k+1, "Q")
                array_payload(v, 3, n, 1, 10**9)
            elif task_id == "3790":
                n, repetitions = v[:2]
                bounded((n,), 1, 100, "n")
                bounded((repetitions,), 1, 10**7, "T")
                array_payload(v, 2, n, 1, 300)
            elif task_id == "3798":
                require(len(v) == 2, "expected n and s")
                bounded(v, 1, 10**11, "n or s")
            elif task_id == "3801":
                n, visits = v[:2]
                bounded((n, visits), 1, 50, "n or m")
                require(len(v) == 2 + 2*n, "wrong like/weight count")
                likes, weights = v[2:2+n], v[2+n:]
                bounded(likes, 0, 1, "like flag")
                require(1 in likes, "no liked picture")
                bounded(weights, 1, 50, "weight")
            elif task_id == "3825":
                require(len(v) == 1, "expected one n")
                bounded(v, 1, 10**9, "n")
            elif task_id == "3868":
                n, m, k = v[:3]
                bounded((n,), 1, 100000, "n")
                bounded((m,), 0, 100000, "m")
                bounded((k,), 1, 10**6, "k")
                require(len(v) == 3 + 4*m, "wrong flight count")
                for offset in range(3, len(v), 4):
                    day, origin, destination, cost = v[offset:offset+4]
                    bounded((day, cost), 1, 10**6, "flight day or cost")
                    bounded((origin, destination), 0, n, "flight city")
                    require((origin == 0) != (destination == 0), "flight needs exactly one metropolis endpoint")
            elif task_id == "3892":
                n, m = v[:2]
                bounded((n,), 2, 100, "n")
                bounded((m,), 1, 200, "m")
                require(len(v) == 2 + 2*m, "wrong candy-route count")
                for origin, destination in zip(v[2::2], v[3::2]):
                    bounded((origin, destination), 1, n, "candy station")
                    require(origin != destination, "candy origin equals destination")
            elif task_id == "3915":
                require(len(v) == 2, "expected p and q")
                p, q = v
                bounded(v, 1, 10000, "p or q")
                require(q <= 42*p and p <= 42*q, "p/q outside1/42..42")
            elif task_id == "3929":
                require(len(v) == 2, "expected N and K")
                n, k = v
                bounded((n,), 1, 2000, "N")
                bounded((k,), 1, n, "K")
            elif task_id == "3957":
                n, k = v[:2]
                bounded((n,), 2, 200000, "n")
                bounded((k,), 1, n//2, "k")
                require(len(v) == 2 + 2*k + 2*(n-1), "wrong university/road count")
                universities = v[2:2+2*k]
                bounded(universities, 1, n, "university vertex")
                require(len(set(universities)) == 2*k, "duplicate university")
                components = graph_components(n, v[2+2*k:])
                require(len(set(components)) == 1, "roads are disconnected")
            elif task_id == "3977":
                n, m, k = v[:3]
                bounded((n,), 1, 1000, "n")
                bounded((m,), 0, 100000, "m")
                bounded((k,), 1, n, "k")
                require(len(v) == 3 + k + 2*m, "wrong government/edge count")
                governments = v[3:3+k]
                bounded(governments, 1, n, "government vertex")
                require(len(set(governments)) == k, "duplicate government")
                components = graph_components(n, v[3+k:])
                require(len({components[g-1] for g in governments}) == k, "government nodes connected")
            elif task_id == "3999":
                n = v[0]
                bounded((n,), 6, 400, "N")
                require(len(v) == 1 + 4*n, "wrong color count")
                bounded(v[1:], 0, 999, "color")
        return {"status": "valid", "reason": "all stated input constraints checked"}
    except DomainInvalid as error:
        return {"status": "invalid", "reason": str(error)}
    except (ValueError, IndexError):
        return {"status": "invalid", "reason": "missing or malformed required integer fields"}


In [ ]:
CASES = {
    "3694": ("3\n0 0 0\n", "1\n-1\n"),
    "3733": ("1 1\n0\n", "1 0\n0\n"),
    "3748": ("1 1\na\n", "1 1\nA\n"),
    "3771": ("2 2\nST\n..\n", "2 2\nSS\nT.\n"),
    "3782": ("1 1 1\n1\n", "2 2 2\n1 1\n"),
    "3790": ("1 1\n300\n", "1 1\n301\n"),
    "3798": ("1\n100000000000\n", "0\n1\n"),
    "3801": ("1 1\n1\n1\n", "1 1\n0\n1\n"),
    "3825": ("1000000000\n", "0\n"),
    "3868": ("1 0 1\n", "1 1 1\n1 0 0 1\n"),
    "3892": ("2 2\n1 2\n1 2\n", "2 1\n1 1\n"),
    "3897": ("1\n1\n", "1\n1000000001\n"),
    "3901": ("1\n1\n", "2\n1\n"),
    "3902": ("abcde\n", "abcd\n"),
    "3915": ("1 42\n", "1 43\n"),
    "3929": ("2000 2000\n", "2 3\n"),
    "3957": ("2 1\n1 2\n1 2\n", "3 1\n1 3\n1 2\n2 1\n"),
    "3977": ("3 1 2\n1 3\n1 2\n", "3 2 2\n1 3\n1 2\n2 3\n"),
    "3999": ("6\n" + "0 0 0 0\n"*6, "6\n" + "1000 0 0 0\n"*6),
}
assert set(CASES) == set(RULES) == {task.task_id for task in dataset.train}
for task_id, (good, bad) in CASES.items():
    assert check_domain(task_id, good)["status"] == "valid", (task_id, check_domain(task_id, good))
    assert check_domain(task_id, bad)["status"] == "invalid", (task_id, check_domain(task_id, bad))
    assert check_domain(task_id, "")["status"] == "invalid"
EXTRA = [
    ("3957", "3 1\n1 1\n1 2\n2 3\n", "invalid"),
    ("3957", "4 1\n1 4\n1 2\n2 3\n3 1\n", "invalid"),
    ("3977", "2 1 1\n1\n1 1\n", "invalid"),
    ("3977", "3 2 1\n1\n1 2\n2 1\n", "invalid"),
    ("3977", "3 0 2\n1 1\n", "invalid"),
    ("3977", "3 3 1\n1\n1 2\n2 3\n3 1\n", "valid"),
    ("3915", "42 1\n", "valid"), ("3915", "43 1\n", "invalid"),
    ("3915", " 13 \t 11\n", "valid"),
    ("3748", "2 2\nab\na\n", "invalid"),
    ("3771", "2 2\nS.\n..\n", "invalid"),
    ("3999", "6\n" + "999 999 999 999\n"*6, "valid"),
]
for task_id, value, expected in EXTRA:
    assert check_domain(task_id, value)["status"] == expected
assert check_domain("unknown", "1")["status"] == "unresolved"
print({"per_task_positive_negative_cases": len(CASES)*2, "empty_cases": len(CASES),
       "additional_cases": len(EXTRA), "unknown_task_case": 1})


In [ ]:
candidate_reviews = {}
totals = Counter()
task_counts = {task.task_id: Counter() for task in dataset.train}
issues = []
for row in rows:
    candidate_id, task_id = row["candidate_id"], row["task_id"]
    checked = []
    for index, value in enumerate(row["inputs"]):
        verdict = check_domain(task_id, value)
        item = {"input_index": index, "input_sha256": hashlib.sha256(value.encode()).hexdigest(), **verdict}
        checked.append(item)
        totals[verdict["status"]] += 1
        task_counts[task_id][verdict["status"]] += 1
        if verdict["status"] != "valid":
            issues.append({"task_id": task_id, "candidate_id": candidate_id, **item})
    candidate_reviews[candidate_id] = {
        "task_id": task_id, "inputs": checked,
        "valid_indices": [x["input_index"] for x in checked if x["status"] == "valid"],
        "invalid_indices": [x["input_index"] for x in checked if x["status"] == "invalid"],
        "unresolved_indices": [x["input_index"] for x in checked if x["status"] == "unresolved"],
        "raw_unique_inputs": len(set(row["inputs"])),
        "whitespace_normalized_unique_inputs": len({tuple(value.split()) for value in row["inputs"]}),
    }
assert sum(totals.values()) == 380
notebook_path = Path("notebooks/azure_pbt_domain_review.ipynb")
report = {
    "scope": "19 frozen training tasks;38 candidates;380 generated inputs",
    "method": "pure statement-derived validators;no model or candidate execution;no outcome filtering",
    "numeric_whitespace_policy": "arbitrary whitespace between integer tokens; grid rows remain contiguous",
    "source_dataset_sha256": hashlib.sha256(DATA.read_bytes()).hexdigest(),
    "source_records_sha256": hashlib.sha256((Path("runs")/RUN/"records.jsonl").read_bytes()).hexdigest(),
    "validator_notebook_sha256": hashlib.sha256(notebook_path.read_bytes()).hexdigest(),
    "task_specification_sha256": {t.task_id: hashlib.sha256(t.specification.encode()).hexdigest() for t in dataset.train},
    "rules": RULES,
    "totals": {status: totals[status] for status in ("valid", "invalid", "unresolved")},
    "per_task": {tid: {status: counts[status] for status in ("valid", "invalid", "unresolved")}
                 for tid, counts in task_counts.items()},
    "candidates": candidate_reviews, "issues": issues,
}
destination = Path("runs")/RUN/"domain-review-v2.json"
if destination.exists():
    assert json.loads(destination.read_text(encoding="utf-8")) == report, "Existing review differs; preserve and version it"
else:
    destination.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(json.dumps({"totals": report["totals"], "per_task": report["per_task"], "issues": issues}, indent=2))
